
# RECIPE GEOFISIKA: ANALISIS MIKROTREMOR DENGAN METODE HVSR

**Nama:** RAHMAT HIDAYAT  
**NIM:** H061231087  
**Kelas:** KECERDASAN DAN ARTIFISIAL UNTUK GEOFISIKA (B)

---

## Recipe ID: GEO-HVSR-001

**Bidang:** Geofisika near-surface dan seismologi  
**Topik:** Estimasi frekuensi fundamental (\(f_0\)) dan rasio HVSR pada puncak (\(A_0\)) dari data mikrotremor tiga komponen.

> **Petunjuk notebook:** Sel bertipe *Markdown* berisi penjelasan/laporan. Sel bertipe *Code* berisi kode Python yang dijalankan dengan tombol **Run**. Jalankan sel secara berurutan dari atas ke bawah.

**Catatan:** Notebook dapat dijalankan menggunakan data simulasi agar tidak memerlukan file lapangan. Jika memiliki data pengukuran asli, ubah pilihan pada bagian pengaturan dan siapkan CSV dengan kolom `time`, `NS`, `EW`, dan `Z`.



## 1. Problem — Permasalahan

Data mikrotremor merekam getaran tanah dalam tiga komponen: horizontal utara–selatan (NS), horizontal timur–barat (EW), dan vertikal (Z). Metode *Horizontal-to-Vertical Spectral Ratio* (HVSR) digunakan untuk membandingkan spektrum komponen horizontal terhadap spektrum komponen vertikal.

Tujuan recipe ini adalah:
1. Memeriksa struktur dan kualitas dasar data.
2. Menghitung spektrum setiap komponen menggunakan metode Welch.
3. Menghitung kurva HVSR.
4. Mengidentifikasi puncak kurva sebagai estimasi awal \(f_0\) dan rasio pada puncak \(A_0\).
5. Menyimpan hasil dan membuat grafik.

**Batasan:** Puncak otomatis yang ditemukan program adalah estimasi awal. Penetapan \(f_0\) yang layak untuk penelitian harus mempertimbangkan pemilihan jendela sinyal, gangguan transien, kestabilan puncak, kriteria kualitas HVSR, serta kondisi geologi setempat.



## 2. Minimal Theory — Teori Singkat

Rasio HVSR dihitung dari spektrum amplitudo tiga komponen:

\[
HVSR(f)=\frac{\sqrt{\left(A_{NS}^{2}(f)+A_{EW}^{2}(f)\right)/2}}{A_Z(f)}
\]

Keterangan:
- \(A_{NS}(f)\): spektrum amplitudo komponen NS.
- \(A_{EW}(f)\): spektrum amplitudo komponen EW.
- \(A_Z(f)\): spektrum amplitudo komponen vertikal.
- \(f\): frekuensi dalam Hz.

Dalam notebook ini, spektrum daya (PSD) dihitung dengan metode Welch, kemudian diakarkan untuk memperoleh besaran spektrum amplitudo yang konsisten. Frekuensi pada puncak kurva disebut \(f_0\), sedangkan nilai kurva pada frekuensi itu disebut \(A_0\).

**Catatan metodologis:** Ini adalah implementasi pembelajaran sederhana, bukan keseluruhan prosedur HVSR standar untuk publikasi atau penilaian kerentanan seismik.



## 3. Persiapan — Library yang Digunakan

Kode berikut mengimpor pustaka untuk pengolahan tabel, perhitungan numerik, analisis spektrum, dan visualisasi.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch, find_peaks
from pathlib import Path

print("Library berhasil diimpor.")



## 4. Prompt — Instruksi Analisis untuk AI

Gunakan instruksi berikut sebagai panduan ketika meminta AI membantu menganalisis data:

> Saya sedang mengolah data mikrotremor dengan metode HVSR. Periksa kelengkapan data dan interval sampling, hitung spektrum tiap komponen dengan metode Welch, hitung kurva HVSR, identifikasi puncak kurva sebagai estimasi awal \(f_0\) dan \(A_0\), buat grafik, dan ekspor hasil ke CSV. Jangan mengarang hasil pengukuran. Jika data simulasi digunakan, nyatakan dengan jelas bahwa hasilnya bukan data lapangan.



## 5. Code — Pengaturan Sumber Data

Pilih sumber data:

- `USE_SIMULATED_DATA = True`: membuat data simulasi untuk memastikan notebook bisa langsung dijalankan.
- `USE_SIMULATED_DATA = False`: membaca file pengukuran bernama `mikrotremor.csv` dari folder kerja notebook.

Untuk data asli, file CSV harus memiliki kolom `time`, `NS`, `EW`, dan `Z`. Kolom `time` harus berupa waktu numerik dalam detik dengan interval sampling yang seragam.


In [ ]:

# Ubah menjadi False jika ingin menggunakan data pengukuran asli.
USE_SIMULATED_DATA = True

if USE_SIMULATED_DATA:
    # Data simulasi untuk demonstrasi; BUKAN hasil pengukuran lapangan.
    rng = np.random.default_rng(42)
    fs_sim = 100.0          # sampling rate (Hz)
    duration_sim = 120.0    # durasi rekaman (detik)
    time = np.arange(0, duration_sim, 1 / fs_sim)

    # Getaran simulasi dengan energi dominan sekitar 2 Hz.
    # Sinyal dibuat berbeda untuk masing-masing komponen.
    NS = 0.8 * np.sin(2 * np.pi * 2.0 * time) + 0.35 * rng.normal(size=len(time))
    EW = 0.7 * np.sin(2 * np.pi * 2.1 * time + 0.4) + 0.35 * rng.normal(size=len(time))
    Z  = 0.35 * np.sin(2 * np.pi * 2.0 * time + 0.2) + 0.35 * rng.normal(size=len(time))

    df = pd.DataFrame({"time": time, "NS": NS, "EW": EW, "Z": Z})
    print("Menggunakan DATA SIMULASI. Hasilnya bukan hasil pengukuran lapangan.")
else:
    input_path = Path("mikrotremor.csv")
    if not input_path.exists():
        raise FileNotFoundError(
            "File mikrotremor.csv tidak ditemukan. Letakkan file di folder notebook "
            "atau ubah input_path ke lokasi file yang benar."
        )
    df = pd.read_csv(input_path)
    print(f"Membaca data dari: {input_path.resolve()}")

df.head()



## 6. Code — Pemeriksaan Dasar Data

Bagian ini memeriksa kolom wajib, nilai kosong, nilai numerik, urutan waktu, dan keseragaman interval sampling. Jika pemeriksaan penting gagal, program menghentikan analisis agar masalah data tidak tersembunyi.


In [ ]:

required_columns = ["time", "NS", "EW", "Z"]

missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Kolom berikut tidak ditemukan: {missing_columns}")

df = df[required_columns].apply(pd.to_numeric, errors="coerce")

print("Jumlah baris dan kolom:", df.shape)
print("\nJumlah nilai kosong per kolom:")
print(df.isna().sum())

if df.isna().any().any():
    raise ValueError("Data mengandung nilai kosong/non-numerik. Bersihkan data sebelum analisis.")

if len(df) < 32:
    raise ValueError("Data terlalu pendek untuk analisis spektrum sederhana.")

time = df["time"].to_numpy(dtype=float)
signals = df[["NS", "EW", "Z"]].to_numpy(dtype=float)

dt = np.diff(time)
if np.any(dt <= 0):
    raise ValueError("Kolom time harus meningkat dan tidak boleh memiliki waktu duplikat.")

median_dt = np.median(dt)
if not np.allclose(dt, median_dt, rtol=0.01, atol=1e-9):
    raise ValueError("Interval sampling tidak seragam (toleransi 1%). Periksa data.")

fs = 1.0 / median_dt
nyquist = fs / 2.0

print(f"\nDurasi rekaman: {time[-1] - time[0]:.2f} detik")
print(f"Interval sampling median: {median_dt:.6f} detik")
print(f"Sampling rate: {fs:.3f} Hz")
print(f"Frekuensi Nyquist: {nyquist:.3f} Hz")



## 7. Code — Menghitung Spektrum dengan Metode Welch

Metode Welch membagi sinyal menjadi beberapa segmen, menghitung spektrum pada setiap segmen, lalu merata-ratakannya. Pada contoh ini digunakan PSD (`power spectral density`) dari tiap komponen.


In [ ]:

# Panjang segmen maksimal 20 detik, atau sepanjang data jika lebih pendek.
nperseg = min(len(df), max(16, int(fs * 20)))

freq, psd_ns = welch(
    signals[:, 0], fs=fs, nperseg=nperseg, detrend="constant"
)
_, psd_ew = welch(
    signals[:, 1], fs=fs, nperseg=nperseg, detrend="constant"
)
_, psd_z = welch(
    signals[:, 2], fs=fs, nperseg=nperseg, detrend="constant"
)

# PSD diakarkan agar diperoleh besaran spektrum amplitudo.
amp_ns = np.sqrt(np.maximum(psd_ns, 0))
amp_ew = np.sqrt(np.maximum(psd_ew, 0))
amp_z = np.sqrt(np.maximum(psd_z, 0))

plt.figure(figsize=(10, 5))
plt.plot(freq, amp_ns, label="NS")
plt.plot(freq, amp_ew, label="EW")
plt.plot(freq, amp_z, label="Z")
plt.xlabel("Frekuensi (Hz)")
plt.ylabel("Akar PSD (satuan amplitudo/√Hz)")
plt.title("Spektrum Tiga Komponen Mikrotremor")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()



## 8. Code — Menghitung Kurva HVSR

Kurva dihitung menggunakan akar rata-rata PSD kedua komponen horizontal dibagi akar PSD komponen vertikal. Frekuensi nol dan nilai vertikal nol dikeluarkan dari perhitungan untuk menghindari pembagian dengan nol.

Rentang analisis di bawah adalah contoh awal 0,1 Hz sampai 20 Hz, dibatasi juga oleh frekuensi Nyquist. Sesuaikan rentang ini dengan tujuan penelitian dan kemampuan instrumen.


In [ ]:

horizontal_amp = np.sqrt((psd_ns + psd_ew) / 2.0)
vertical_amp = np.sqrt(np.maximum(psd_z, 0))

valid = (freq > 0) & (vertical_amp > 0)
freq_valid = freq[valid]
hvsr_valid = horizontal_amp[valid] / vertical_amp[valid]

f_min = 0.1
f_max = min(20.0, nyquist)
band = (freq_valid >= f_min) & (freq_valid <= f_max)

f = freq_valid[band]
hvsr = hvsr_valid[band]

if len(f) < 3:
    raise ValueError(
        "Terlalu sedikit titik frekuensi pada rentang analisis. "
        "Periksa durasi data, sampling rate, atau rentang frekuensi."
    )

print(f"Rentang frekuensi yang dianalisis: {f.min():.3f}–{f.max():.3f} Hz")



## 9. Code — Estimasi Puncak \(f_0\) dan \(A_0\)

Program mengidentifikasi puncak lokal pada kurva HVSR. Jika tidak ada puncak lokal, program menggunakan titik maksimum dalam rentang yang dianalisis. Cara otomatis ini hanya menghasilkan kandidat puncak; validitasnya perlu diperiksa lebih lanjut.


In [ ]:

peaks, _ = find_peaks(hvsr)

if len(peaks) > 0:
    peak_idx = peaks[np.argmax(hvsr[peaks])]
    peak_method = "puncak lokal tertinggi"
else:
    peak_idx = int(np.argmax(hvsr))
    peak_method = "maksimum pada rentang analisis (tidak ditemukan puncak lokal)"

f0 = float(f[peak_idx])
A0 = float(hvsr[peak_idx])

print("=== RINGKASAN HASIL HVSR ===")
print(f"Metode pemilihan puncak : {peak_method}")
print(f"Estimasi frekuensi f0  : {f0:.3f} Hz")
print(f"Rasio HVSR pada puncak : {A0:.3f}")
print("\nPENTING: hasil ini adalah estimasi otomatis awal, bukan validasi HVSR final.")



## 10. Code — Visualisasi Kurva HVSR

Grafik berikut menampilkan kurva HVSR serta penanda pada puncak yang dipilih secara otomatis.


In [ ]:

plt.figure(figsize=(10, 5))
plt.plot(f, hvsr, label="Kurva HVSR")
plt.axvline(f0, linestyle="--", label=f"f0 = {f0:.2f} Hz")
plt.scatter([f0], [A0], zorder=3, label=f"A0 = {A0:.2f}")
plt.xlabel("Frekuensi (Hz)")
plt.ylabel("Rasio HVSR")
plt.title("Analisis Mikrotremor dengan Metode HVSR")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()



## 11. Code — Menyimpan Hasil ke CSV

Hasil kurva HVSR disimpan sebagai `hasil_hvsr.csv`. Ringkasan parameter puncak juga disimpan sebagai `ringkasan_hvsr.csv`. File akan dibuat di folder kerja notebook.


In [ ]:

result = pd.DataFrame({
    "frequency_Hz": f,
    "HVSR": hvsr
})
result.to_csv("hasil_hvsr.csv", index=False)

summary = pd.DataFrame([{
    "peak_method": peak_method,
    "f0_Hz": f0,
    "A0_HVSR": A0,
    "sampling_rate_Hz": fs,
    "data_type": "simulasi" if USE_SIMULATED_DATA else "pengukuran"
}])
summary.to_csv("ringkasan_hvsr.csv", index=False)

print("File tersimpan:")
print("- hasil_hvsr.csv")
print("- ringkasan_hvsr.csv")
display(summary)



## 12. Result — Hasil dan Interpretasi

Setelah seluruh sel kode dijalankan, perhatikan:
- Grafik spektrum tiga komponen.
- Grafik kurva HVSR.
- Nilai estimasi \(f_0\) dan \(A_0\) pada output.
- File CSV yang berisi kurva dan ringkasan.

**Jika menggunakan data simulasi:** hasil hanya menunjukkan bahwa alur komputasi dapat dijalankan. Nilai yang diperoleh tidak boleh dilaporkan sebagai hasil pengukuran lapangan dan tidak dapat digunakan untuk menyimpulkan kondisi tanah di lokasi nyata.

**Jika menggunakan data lapangan:** interpretasikan nilai dengan mempertimbangkan kualitas kurva, konsistensi puncak antarjendela, durasi rekaman, instrumen, kondisi geologi, dan kriteria HVSR yang relevan. Nilai \(A_0\) bukan secara otomatis faktor amplifikasi absolut tanah.



## 13. Optional Check — Checklist Kualitas

Gunakan daftar periksa berikut sebelum memakai hasil dalam laporan.

- [ ] Kolom data lengkap dan tidak memiliki nilai kosong.
- [ ] Waktu berurutan dan interval sampling seragam.
- [ ] Gangguan transien/noise diperiksa dan ditangani secara tepat.
- [ ] Rentang frekuensi sesuai dengan tujuan analisis dan kemampuan instrumen.
- [ ] Puncak HVSR diperiksa kestabilannya pada beberapa jendela sinyal.
- [ ] Interpretasi tidak hanya bergantung pada satu nilai puncak.
- [ ] Data simulasi dan data pengukuran dibedakan secara eksplisit.
- [ ] Hasil didukung informasi geologi atau data lain jika digunakan untuk interpretasi tapak.

## 14. Kesimpulan

Recipe GEO-HVSR-001 menyusun alur dasar analisis data mikrotremor, mulai dari pemeriksaan data, perhitungan spektrum, perhitungan HVSR, penentuan kandidat puncak, visualisasi, hingga ekspor hasil. Notebook ini dapat digunakan sebagai latihan pengolahan data yang dapat diulang (*reproducible workflow*), dengan catatan bahwa validasi ilmiah tetap diperlukan sebelum menarik kesimpulan geofisika.
